In [1]:
#conda activate burnseverity
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import requests
import json

import geopandas as gpd

import rasterio as rio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from rasterio import features
from rasterio.plot import show_hist

from shapely.geometry import shape, mapping
from shapely.ops import unary_union

import validation as val

from src import config
from src import process


### Validation goals

There a four **questions** we want to test:
1. Do our boundaries and those from CalFire match?
2. How do the two different indices (dNBR and RBR) calculated with our tool differ?
3. How does the time window chosed affect the resultts
4. For fires, where this is available,how does the dNBR maps from our tool compare to those from BAER?

We will use the following **metrics** to test these questions
1. Percent overlap between Calfire shapefile and vectorized pixels with $x > 0$
2. Compare  mean, spread, SD, and interquartile range dNBR and RBR
3. Compare mean and variance for all time windows tested and the two reference points (alarm date vs. containment date)
4. $R^2$ on a pixel basis between the two images 

This notebook calculates these metrics where they require **post-processing tasks**
1. Filter for pixels with $x > 0$ and convert to a shapefile, calculate overlap with Calfire boundary
2. & 3. Calculate mean, median, SD, spread, and interquartile range of pixels within Calfire boundaries
4. Recoarse Sentinel to Landsat, calculate $R^2$ based on Calfire boundary

In [2]:
fires = pd.read_csv(config.PATH_JOBS_LOG)
calfire = gpd.read_file(config.PATH_FIRES_VALIDATION)
fire_names = calfire['FIRE_NAME'].unique()

In [ ]:
indicators = []

for fire_name in ['GEOLOGY']: #fire_names:
  
    calfire_polygon = calfire[calfire['FIRE_NAME'] == fire_name]

    for metric in config.METRICS:

        print(f'Processing fire: {fire_name}, {metric} ...')

        for post_fire_reference_point in config.POST_FIRE_REFERENCE_POINT:

            for post_fire_period in config.POST_FIRE_PERIOD:

                fire_event_name = fires.loc[(fires['fire_name'] == fire_name) & 
                                (fires['post_fire_days'] == post_fire_period) &
                                (fires['date_mode'] == post_fire_reference_point), 'fire_event_name'].values[0]

                url_raster = process.get_url_raster(fires, fire_name, post_fire_period, post_fire_reference_point, metric)
                raster = process.get_raster_as_lonlat(url_raster)[0]
                polygon_from_raster = process.convert_burnscar_to_polygon(raster)
                polygon_from_raster.to_file(f'output/{fire_event_name}_from_{metric}.shp')

                mean, var, max, min, q_25, q_50, q_75 = process.calculate_statistical_indicators(raster, calfire_polygon)

                process.append_results(fire_name, post_fire_reference_point, post_fire_period, metric, 
                                       mean, var, max, min, q_25, q_50, q_75, indicators)
                

df_indicators = pd.DataFrame(indicators)
df_indicators.to_csv(config.PATH_INDICATORS, index=False)


                



TypeError: append_results() takes 12 positional arguments but 13 were given

In [4]:
# Process all fires, date modes, and date ranges
results_list = []
current_key = None

for _, row in fires.iterrows():
    key = (row['fire_name'], row['date_mode'])
    if key != current_key:
        print(f"Processing {row['fire_name']} ({row['date_mode']})...")
        current_key = key

    result = val.process_fire_metrics(row['fire_name'], row['post_fire_days'], row['date_mode'], fires, calfire)

    if result is False:
        continue

    # result is now a list of dicts (one per metric)
    results_list.extend(result)

# Create DataFrame
metrics_df = pd.DataFrame(results_list)

# Split by metric and save separate geopackages
dnbr_df = metrics_df[metrics_df['metric'] == 'dnbr'].copy()
rbr_df = metrics_df[metrics_df['metric'] == 'rbr'].copy()

# Save dNBR as GeoPackage
dnbr_gdf = gpd.GeoDataFrame(
    dnbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
dnbr_gdf.to_file('validation_metrics_dnbr.gpkg', driver='GPKG')

# Save RBR as GeoPackage
rbr_gdf = gpd.GeoDataFrame(
    rbr_df,
    geometry='filtered_polygon',
    crs='EPSG:4326'
)
rbr_gdf.to_file('validation_metrics_rbr.gpkg', driver='GPKG')

# Also save combined attributes as CSV for quick viewing
metrics_df.drop('filtered_polygon', axis=1).to_csv('validation_metrics.csv', index=False)

print(f"\nFinal table shape: {metrics_df.shape}")
print(f"dNBR rows: {len(dnbr_df)}, RBR rows: {len(rbr_df)}")
print("Saved spatial data to validation_metrics_dnbr.gpkg and validation_metrics_rbr.gpkg")
print("Saved CSV to validation_metrics.csv")

metrics_df.head()


Processing COFFEE POT (alarm)...
Processing COFFEE POT (cont)...
Processing SENTINEL (alarm)...
Processing SENTINEL (cont)...
Processing SIMPSON (alarm)...
Processing SIMPSON (cont)...
Processing YORK (alarm)...
  Job YORK_date2023-07-28_range5_modealarm is failed
  Job YORK_date2023-07-28_range10_modealarm is failed
  Job YORK_date2023-07-28_range15_modealarm is failed
  Job YORK_date2023-07-28_range21_modealarm is failed
  Job YORK_date2023-07-28_range30_modealarm is failed
  Job YORK_date2023-07-28_range45_modealarm is failed
  Job YORK_date2023-07-28_range60_modealarm is failed
  Job YORK_date2023-07-28_range90_modealarm is failed
Processing YORK (cont)...
  Job YORK_date2023-08-20_range5_modecont is failed
  Job YORK_date2023-08-20_range10_modecont is failed
  Job YORK_date2023-08-20_range15_modecont is failed
  Job YORK_date2023-08-20_range21_modecont is failed
  Job YORK_date2023-08-20_range30_modecont is failed
  Job YORK_date2023-08-20_range45_modecont is failed
  Job YORK_dat

,fire_name,fire_days,date_mode,fire_event_name,metric,calfire_mean,calfire_var,calfire_n_removed,calfire_pct_removed,filtered_mean,filtered_var,filtered_n_removed,filtered_pct_removed,filtered_polygon
0,COFFEE POT,5,alarm,COFFEE POT_date2024-08-03_range5_modealarm,dnbr,0.057042,0.005021,130,0.074220,0.075088,0.004026,130,0.089564,MULTIPOLYGON (((-118.78054641229755 36.3526655...
1,COFFEE POT,5,alarm,COFFEE POT_date2024-08-03_range5_modealarm,rbr,0.030835,0.001340,0,0.000000,0.040979,0.000960,0,0.000000,MULTIPOLYGON (((-118.78054641229755 36.3526655...
2,COFFEE POT,10,alarm,COFFEE POT_date2024-08-03_range10_modealarm,dnbr,0.054348,0.004790,109,0.062231,0.073521,0.003462,106,0.073420,MULTIPOLYGON (((-118.7948966008226 36.35266556...
3,COFFEE POT,10,alarm,COFFEE POT_date2024-08-03_range10_modealarm,rbr,0.030574,0.001393,0,0.000000,0.041394,0.000942,0,0.000000,MULTIPOLYGON (((-118.7948966008226 36.35266556...
4,COFFEE POT,15,alarm,COFFEE POT_date2024-08-03_range15_modealarm,dnbr,0.027727,0.005874,96,0.054809,0.063458,0.003400,81,0.067669,MULTIPOLYGON (((-118.79507597817916 36.3524830...
